# release16 risk-gated state-head objective screen

Baselines: canonical `soxlbacktest_qlib_walkforward_baseline_v1`; direct comparison `release16_state_head_revision_handoff_v1` / `release16_failedfold_state_head_checkpoint_bc_screen_v1`.

Role: diagnose and build the next objective screen before any retained actor training.

Target window: official walk-forward folds `wf_2020`, `wf_2022`, `wf_2023_2024`.

Changed layer: `model_selection` / checkpoint objective and state-head target gating.

External reference: the repo-owned SOXLBacktest qlib reference baseline and official daily-path replay artifacts.

Hypothesis: a risk-gated state-head target can preserve the state-head upside lift while routing drawdown-drift intervals back to the safer participation-overlay path. Acceptance requires every fold to clear `relative_return >= 30pp` and `MaxDD < 20%`; otherwise the next step must train a new non-convex objective rather than retain or extend the existing actor.

Closest success case: `release16_failedfold_overlay_checkpoint_screen_v1` lowered drawdown in the failed folds versus the state-head checkpoint.

Closest failure case: `release16_failedfold_state_head_checkpoint_bc_screen_v1` kept more upside but failed MaxDD in `wf_2020` and `wf_2023_2024`; `release16_failedfold_overlay_checkpoint_screen_v1` controlled drawdown but missed the relative-return target.

Why this is not a duplicate: this screen evaluates risk-gated combinations of the two official daily paths and emits a checkpoint-objective recipe; it is not another threshold tweak on a single replay output.


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'SPEC.md').exists())
CARD = ROOT / 'runtime/model_selection/card/release16_risk_gated_state_head_objective_screen_v1'
STATE_CARD = ROOT / 'runtime/model_selection/card/release16_failedfold_state_head_checkpoint_bc_screen_v1'
OVERLAY_CARD = ROOT / 'runtime/model_selection/card/release16_failedfold_overlay_checkpoint_screen_v1'
HANDOFF_CARD = ROOT / 'runtime/model_selection/card/release16_state_head_revision_handoff_v1'
FRONTIER_CARD = ROOT / 'runtime/model_selection/card/model_retention_frontier_v1'
GATE_CARD = ROOT / 'runtime/model_selection/card/model_retention_walkforward_gate_v1'

WINDOWS = ['wf_2020', 'wf_2022', 'wf_2023_2024']
TARGET_RELATIVE_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
INITIAL_NAV = 1_000_000.0

print('card', CARD.relative_to(ROOT))
print('handoff', HANDOFF_CARD.relative_to(ROOT))


card runtime/model_selection/card/release16_risk_gated_state_head_objective_screen_v1
handoff runtime/model_selection/card/release16_state_head_revision_handoff_v1


In [2]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

handoff = load_json(HANDOFF_CARD / 'summary.json')
state_summary = load_json(STATE_CARD / 'summary.json')
overlay_summary = load_json(OVERLAY_CARD / 'summary.json')
frontier_summary = load_json(FRONTIER_CARD / 'summary.json').get('summary', {})
gate_summary = load_json(GATE_CARD / 'summary.json').get('summary', {})

print('frontier recommended', frontier_summary.get('recommended_next_source'), frontier_summary.get('recommended_next_action'))
print('handoff decision', handoff.get('decision'))
print('state failed folds', state_summary.get('failed_folds'))
print('overlay failed folds', overlay_summary.get('failed_folds'))


frontier recommended release16_state_head_revision_handoff_v1 build_risk_gated_state_head_target_or_drawdown_drift_checkpoint_objective_then_rerun_official_three_fold_replay
handoff decision state_head_revision_handoff_no_training_no_actor_retention
state failed folds ['wf_2020', 'wf_2022', 'wf_2023_2024']
overlay failed folds ['wf_2020', 'wf_2023_2024']


In [3]:
def daily_path(card: Path, fold: str) -> pd.DataFrame:
    path = card / 'official_daily_paths' / f'{fold}__bc_initialization_official_daily_path.csv'
    frame = pd.read_csv(path, parse_dates=['datetime']).sort_values('datetime')
    frame['daily_return'] = frame['strategy_nav'].astype(float).pct_change().fillna(0.0)
    frame['price_return'] = frame['price'].astype(float).pct_change().fillna(0.0)
    frame['buyhold_return'] = frame['buyhold_nav'].astype(float).pct_change().fillna(0.0)
    return frame

paths = {}
for fold in WINDOWS:
    state = daily_path(STATE_CARD, fold).add_prefix('state_')
    overlay = daily_path(OVERLAY_CARD, fold).add_prefix('overlay_')
    joined = state.merge(
        overlay,
        left_on='state_datetime',
        right_on='overlay_datetime',
        how='inner',
        validate='one_to_one',
    )
    joined = joined.rename(columns={'state_datetime': 'datetime'}).drop(columns=['overlay_datetime'])
    joined['benchmark_return'] = joined['state_buyhold_return']
    paths[fold] = joined
    print(fold, joined.shape, joined['datetime'].min().date(), joined['datetime'].max().date())


wf_2020 (252, 22) 2020-01-02 2020-12-30
wf_2022 (250, 22) 2022-01-03 2022-12-29
wf_2023_2024 (251, 22) 2024-01-02 2024-12-30


In [4]:
def metrics_from_returns(frame: pd.DataFrame, returns: pd.Series, position: pd.Series) -> dict:
    nav = (1.0 + returns.fillna(0.0)).cumprod() * INITIAL_NAV
    buyhold_nav = frame['state_buyhold_nav'].astype(float)
    final_return = (float(nav.iloc[-1]) / INITIAL_NAV - 1.0) * 100.0
    buyhold_return = (float(buyhold_nav.iloc[-1]) / float(buyhold_nav.iloc[0]) - 1.0) * 100.0
    drawdown = nav / nav.cummax() - 1.0
    max_dd = abs(float(drawdown.min())) * 100.0
    avg_pos = float(position.fillna(0.0).mean())
    rel = final_return - buyhold_return
    rel_gap = max(0.0, TARGET_RELATIVE_PP - rel)
    dd_gap = max(0.0, max_dd - TARGET_MAX_DD_PCT)
    gate_loss = rel_gap + 2.0 * dd_gap
    return {
        'return_pct': final_return,
        'buyhold_return_pct': buyhold_return,
        'relative_return_pct': rel,
        'max_dd_pct': max_dd,
        'avg_pos': avg_pos,
        'relative_gap_pp': rel_gap,
        'drawdown_gap_pp': dd_gap,
        'gate_loss': gate_loss,
        'relative_gate': bool(rel >= TARGET_RELATIVE_PP),
        'drawdown_gate': bool(max_dd < TARGET_MAX_DD_PCT),
        'gate_pass': bool(rel >= TARGET_RELATIVE_PP and max_dd < TARGET_MAX_DD_PCT),
    }


def build_static_blend(frame: pd.DataFrame, weight: float) -> tuple[pd.Series, pd.Series]:
    returns = weight * frame['state_daily_return'] + (1.0 - weight) * frame['overlay_daily_return']
    position = weight * frame['state_position'].astype(float) + (1.0 - weight) * frame['overlay_position'].astype(float)
    return returns, position


def build_drawdown_fallback(frame: pd.DataFrame, threshold: float, cooldown: int) -> tuple[pd.Series, pd.Series]:
    state_ret = frame['state_daily_return'].astype(float).reset_index(drop=True)
    overlay_ret = frame['overlay_daily_return'].astype(float).reset_index(drop=True)
    state_pos = frame['state_position'].astype(float).reset_index(drop=True)
    overlay_pos = frame['overlay_position'].astype(float).reset_index(drop=True)
    returns = []
    positions = []
    nav = INITIAL_NAV
    peak = INITIAL_NAV
    cooldown_left = 0
    for i in range(len(frame)):
        drawdown = nav / peak - 1.0
        use_overlay = cooldown_left > 0 or drawdown <= -threshold
        r = overlay_ret.iloc[i] if use_overlay else state_ret.iloc[i]
        p = overlay_pos.iloc[i] if use_overlay else state_pos.iloc[i]
        returns.append(float(r))
        positions.append(float(p))
        nav *= 1.0 + float(r)
        peak = max(peak, nav)
        if use_overlay:
            cooldown_left = max(cooldown_left - 1, cooldown)
        else:
            cooldown_left = max(cooldown_left - 1, 0)
    return pd.Series(returns), pd.Series(positions)


def build_position_cap_fallback(frame: pd.DataFrame, cap: float) -> tuple[pd.Series, pd.Series]:
    state_pos = frame['state_position'].astype(float)
    use_overlay = state_pos > cap
    returns = frame['state_daily_return'].where(~use_overlay, frame['overlay_daily_return']).astype(float)
    position = frame['state_position'].where(~use_overlay, frame['overlay_position']).astype(float)
    return returns, position


In [5]:
rows = []
for fold, frame in paths.items():
    for weight in np.linspace(0.0, 1.0, 21):
        returns, position = build_static_blend(frame, float(weight))
        row = metrics_from_returns(frame, returns, position)
        row.update({'fold': fold, 'candidate_id': f'static_blend_state_{weight:.2f}', 'family': 'static_blend', 'state_weight': round(float(weight), 4), 'drawdown_threshold': None, 'cooldown_days': None, 'position_cap': None})
        rows.append(row)
    for threshold in [0.05, 0.075, 0.10, 0.125, 0.15, 0.175, 0.20]:
        for cooldown in [5, 10, 20, 40, 60]:
            returns, position = build_drawdown_fallback(frame, threshold, cooldown)
            row = metrics_from_returns(frame, returns, position)
            row.update({'fold': fold, 'candidate_id': f'dd_fallback_{threshold:.3f}_{cooldown}d', 'family': 'drawdown_fallback', 'state_weight': None, 'drawdown_threshold': threshold, 'cooldown_days': cooldown, 'position_cap': None})
            rows.append(row)
    for cap in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
        returns, position = build_position_cap_fallback(frame, cap)
        row = metrics_from_returns(frame, returns, position)
        row.update({'fold': fold, 'candidate_id': f'position_cap_{cap:.2f}', 'family': 'position_cap_fallback', 'state_weight': None, 'drawdown_threshold': None, 'cooldown_days': None, 'position_cap': cap})
        rows.append(row)

candidate_grid = pd.DataFrame(rows)
for column in ['return_pct','buyhold_return_pct','relative_return_pct','max_dd_pct','avg_pos','relative_gap_pp','drawdown_gap_pp','gate_loss']:
    candidate_grid[column] = candidate_grid[column].astype(float).round(6)

candidate_grid = candidate_grid.sort_values(['fold', 'gate_loss', 'drawdown_gap_pp', 'relative_gap_pp', 'max_dd_pct', 'candidate_id']).reset_index(drop=True)
frontier = candidate_grid.groupby('fold', group_keys=False).head(8).copy()
selected = candidate_grid.groupby('fold', group_keys=False).head(1).copy().reset_index(drop=True)

candidate_grid.to_csv(CARD / 'candidate_grid.csv', index=False)
frontier.to_csv(CARD / 'fold_candidate_frontier.csv', index=False)
selected.to_csv(CARD / 'selected_by_fold.csv', index=False)

selected[['fold','candidate_id','family','return_pct','relative_return_pct','max_dd_pct','avg_pos','relative_gap_pp','drawdown_gap_pp','gate_pass','gate_loss']]


,fold,candidate_id,family,return_pct,relative_return_pct,max_dd_pct,avg_pos,relative_gap_pp,drawdown_gap_pp,gate_pass,gate_loss
0,wf_2020,dd_fallback_0.075_10d,drawdown_fallback,79.426700,21.333130,16.652151,0.307464,8.666870,0.0,False,8.666870
1,wf_2022,static_blend_state_0.80,static_blend,-14.345198,72.242874,15.341329,0.086368,0.000000,0.0,True,0.000000
2,wf_2023_2024,position_cap_0.20,position_cap_fallback,22.429328,22.785963,5.546722,0.097579,7.214037,0.0,False,7.214037


In [6]:
display(frontier[['fold','candidate_id','family','return_pct','relative_return_pct','max_dd_pct','avg_pos','relative_gap_pp','drawdown_gap_pp','gate_pass','gate_loss']])

gate_pass_count = int(selected['gate_pass'].sum())
all_gate_pass = bool(gate_pass_count == len(WINDOWS))
failed_folds = selected.loc[~selected['gate_pass'], 'fold'].tolist()
worst_drawdown = float(selected['max_dd_pct'].max())
min_relative = float(selected['relative_return_pct'].min())
max_relative_gap = float(selected['relative_gap_pp'].max())
max_drawdown_gap = float(selected['drawdown_gap_pp'].max())
print('gate_pass_count', gate_pass_count, 'all_gate_pass', all_gate_pass)
print('failed_folds', failed_folds)
print('worst_drawdown', worst_drawdown, 'min_relative', min_relative, 'max_relative_gap', max_relative_gap, 'max_drawdown_gap', max_drawdown_gap)


,fold,candidate_id,family,return_pct,relative_return_pct,max_dd_pct,avg_pos,relative_gap_pp,drawdown_gap_pp,gate_pass,gate_loss
0,wf_2020,dd_fallback_0.075_10d,drawdown_fallback,79.426700,21.333130,16.652151,0.307464,8.666870,0.000000,False,8.666870
1,wf_2020,dd_fallback_0.075_20d,drawdown_fallback,79.426700,21.333130,16.652151,0.307464,8.666870,0.000000,False,8.666870
2,wf_2020,dd_fallback_0.075_40d,drawdown_fallback,79.426700,21.333130,16.652151,0.307464,8.666870,0.000000,False,8.666870
3,wf_2020,dd_fallback_0.075_5d,drawdown_fallback,79.426700,21.333130,16.652151,0.307464,8.666870,0.000000,False,8.666870
4,wf_2020,dd_fallback_0.075_60d,drawdown_fallback,79.426700,21.333130,16.652151,0.307464,8.666870,0.000000,False,8.666870
5,wf_2020,dd_fallback_0.100_10d,drawdown_fallback,93.070835,34.977264,24.546108,0.361550,0.000000,4.546108,False,9.092216
6,wf_2020,dd_fallback_0.100_20d,drawdown_fallback,93.070835,34.977264,24.546108,0.361550,0.000000,4.546108,False,9.092216
7,wf_2020,dd_fallback_0.100_40d,drawdown_fallback,93.070835,34.977264,24.546108,0.361550,0.000000,4.546108,False,9.092216
65,wf_2022,static_blend_state_0.80,static_blend,-14.345198,72.242874,15.341329,0.086368,0.000000,0.000000,True,0.000000
66,wf_2022,static_blend_state_0.75,static_blend,-14.376031,72.212041,15.356415,0.086057,0.000000,0.000000,True,0.000000


gate_pass_count 1 all_gate_pass False
failed_folds ['wf_2020', 'wf_2023_2024']
worst_drawdown 16.652151 min_relative 21.33313 max_relative_gap 8.66687 max_drawdown_gap 0.0


In [7]:
recipe_rows = [
    {
        'objective_term': 'nonconvex_upside_floor_after_risk_gate',
        'applies_to_folds': ','.join(failed_folds) if failed_folds else '',
        'evidence': 'No static blend, drawdown fallback, or position-cap fallback from existing official paths cleared the relative-return gate on every fold.',
        'implementation_target': 'Train a new checkpoint/reward objective that adds state-head upside only when a risk-state classifier predicts low drawdown drift; do not only interpolate existing paths.',
        'acceptance_gate': 'official three-fold replay relative_return >= 30pp and MaxDD < 20% on wf_2020,wf_2022,wf_2023_2024',
    },
    {
        'objective_term': 'drawdown_drift_stop_loss_in_checkpoint_score',
        'applies_to_folds': 'wf_2020,wf_2023_2024',
        'evidence': f'Best screened candidates still have max drawdown gap {max_drawdown_gap:.2f}pp or relative gap {max_relative_gap:.2f}pp.',
        'implementation_target': 'Score checkpoints by official daily-path gate loss, with drawdown drift weighted before headline return.',
        'acceptance_gate': 'selected checkpoint has gate_loss=0 on all required folds before any retained actor review',
    },
    {
        'objective_term': 'state_head_slice_fit_precondition',
        'applies_to_folds': ','.join(WINDOWS),
        'evidence': 'Source handoff recorded zero strict gate passes and zero BC slice gate passes.',
        'implementation_target': 'Require state-head target slices to pass before PPO continuation; otherwise keep actor_retention_allowed=false.',
        'acceptance_gate': 'BC/state-head slice gates pass before full actor training starts',
    },
]
objective_recipe = pd.DataFrame(recipe_rows)
objective_recipe.to_csv(CARD / 'objective_recipe.csv', index=False)
objective_recipe


,objective_term,applies_to_folds,evidence,implementation_target,acceptance_gate
0,nonconvex_upside_floor_after_risk_gate,"wf_2020,wf_2023_2024","No static blend, drawdown fallback, or positio...",Train a new checkpoint/reward objective that a...,official three-fold replay relative_return >= ...
1,drawdown_drift_stop_loss_in_checkpoint_score,"wf_2020,wf_2023_2024",Best screened candidates still have max drawdo...,Score checkpoints by official daily-path gate ...,selected checkpoint has gate_loss=0 on all req...
2,state_head_slice_fit_precondition,"wf_2020,wf_2022,wf_2023_2024",Source handoff recorded zero strict gate passe...,Require state-head target slices to pass befor...,BC/state-head slice gates pass before full act...


In [8]:
selected_by_fold = {}
metrics = []
for _, row in selected.iterrows():
    fold = str(row['fold'])
    selected_by_fold[fold] = {
        'selected_candidate': str(row['candidate_id']),
        'family': str(row['family']),
        'selected_return': float(row['return_pct']),
        'selected_relative_return': float(row['relative_return_pct']),
        'selected_max_dd': float(row['max_dd_pct']),
        'selected_avg_pos': float(row['avg_pos']),
        'relative_gap_to_target_pp': float(row['relative_gap_pp']),
        'drawdown_gap_to_target_pp': float(row['drawdown_gap_pp']),
        'selected_candidate_gate': bool(row['gate_pass']),
        'gate_loss': float(row['gate_loss']),
    }
    metrics.append({
        'window': fold,
        'split_type': 'walk_forward_test',
        'return_pct': float(row['return_pct']),
        'relative_return_pct': float(row['relative_return_pct']),
        'max_dd_pct': float(row['max_dd_pct']),
        'avg_pos': float(row['avg_pos']),
    })

decision = (
    'risk_gated_state_head_objective_screen_gate_pass_ready_for_official_replay'
    if all_gate_pass
    else 'risk_gated_state_head_objective_screen_no_full_gate_no_actor_retention'
)
next_mechanism = (
    'rerun_official_three_fold_checkpoint_replay_from_risk_gated_state_head_objective'
    if all_gate_pass
    else 'train_nonconvex_risk_gated_state_head_checkpoint_objective_then_rerun_official_three_fold_replay'
)

summary = {
    'experiment': 'release16_risk_gated_state_head_objective_screen_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
    'direct_comparison_baseline': 'release16_state_head_revision_handoff_v1',
    'source_card': 'release16_state_head_revision_handoff_v1',
    'source_handoff_decision': handoff.get('decision'),
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'checkpoint_objective', 'state_head_target_gating'],
    'target_window': WINDOWS,
    'target_relative_return_pp': TARGET_RELATIVE_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'accepted_tradeoff': 'No actor retention unless all official walk-forward folds pass MaxDD and relative-return gates.',
    'hypothesis': 'Risk gating can preserve state-head upside while preventing drawdown drift; this screen tests whether existing official paths are sufficient before launching new training.',
    'acceptance_criteria': 'All selected fold candidates must have relative_return_pct >= 30 and max_dd_pct < 20.',
    'failure_mode': 'Existing state-head/overlay path combinations cannot produce enough upside under the drawdown gate, requiring a newly trained non-convex objective.',
    'closest_historical_success_case': 'release16_failedfold_overlay_checkpoint_screen_v1 reduced drawdown versus the state-head checkpoint in failed folds.',
    'closest_historical_failure_case': 'release16_failedfold_state_head_checkpoint_bc_screen_v1 overexposed failed folds; overlay checkpoint then underparticipated relative to the return target.',
    'why_not_duplicate': 'Screens path-level risk-gated combinations and emits a new objective recipe; it does not tune the same state-head threshold or reuse a single failed replay.',
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'next_training_allowed_after_objective_screen': bool(not all_gate_pass),
    'next_mechanism': next_mechanism,
    'gate_pass_count': gate_pass_count,
    'failed_folds': failed_folds,
    'worst_wf_max_drawdown_pct': worst_drawdown,
    'min_wf_relative_return_pp': min_relative,
    'max_relative_gap_to_target_pp': max_relative_gap,
    'max_drawdown_gap_to_target_pp': max_drawdown_gap,
    'selected_by_fold': selected_by_fold,
    'metrics': metrics,
    'sidecars': {
        'candidate_grid': str((CARD / 'candidate_grid.csv').relative_to(ROOT)),
        'fold_candidate_frontier': str((CARD / 'fold_candidate_frontier.csv').relative_to(ROOT)),
        'selected_by_fold': str((CARD / 'selected_by_fold.csv').relative_to(ROOT)),
        'objective_recipe': str((CARD / 'objective_recipe.csv').relative_to(ROOT)),
        'selection': str((CARD / 'selection.json').relative_to(ROOT)),
        'summary': str((CARD / 'summary.json').relative_to(ROOT)),
    },
}
selection = dict(summary)

(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(CARD / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True) + '\n', encoding='utf-8')
summary


{'experiment': 'release16_risk_gated_state_head_objective_screen_v1',
 'created_at': '2026-06-14T23:55:40.875315+00:00',
 'decision': 'risk_gated_state_head_objective_screen_no_full_gate_no_actor_retention',
 'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'direct_comparison_baseline': 'release16_state_head_revision_handoff_v1',
 'source_card': 'release16_state_head_revision_handoff_v1',
 'source_handoff_decision': 'state_head_revision_handoff_no_training_no_actor_retention',
 'role': 'diagnose',
 'changed_layer': ['model_selection',
  'checkpoint_objective',
  'state_head_target_gating'],
 'target_window': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 20.0,
 'accepted_tradeoff': 'No actor retention unless all official walk-forward folds pass MaxDD and relative-return gates.',
 'hypothesis': 'Risk gating can preserve state-head upside while preventing drawdown drift; this screen tests whether existing official path

## Conclusion

This notebook screens whether existing official state-head and participation-overlay daily paths can be combined into a risk-gated objective that already satisfies the retention gates. If the selected fold candidates do not pass every walk-forward fold, the result is not actor-retention evidence; it is an objective handoff for a new non-convex risk-gated checkpoint/reward training run followed by official three-fold replay.
